## 2 序列模型

### 2.1 理论计算题

给定字符序列 `"ababc"`，词汇表 $\{a,b,c\}$，一阶马尔可夫模型，拉普拉斯平滑（加1平滑）。

统计转移次数（相邻字符对）：
- $a \to b$：2 次（位置1-2，3-4）
- $b \to a$：1 次（位置2-3）
- $b \to c$：1 次（位置4-5）
- 其他转移为 0 次。

以 $b$ 为条件：
- $count(b, \cdot) = count(b,a) + count(b,b) + count(b,c) = 1 + 0 + 1 = 2$
- 词汇表大小 $|V| = 3$

拉普拉斯平滑公式：
$$p(x' | b) = \frac{count(b,x') + 1}{\sum_{x''} count(b,x'') + |V|}$$

计算：
1. $p(a|b) = \frac{1+1}{2+3} = \frac{2}{5} = 0.4$
2. $p(c|b) = \frac{1+1}{2+3} = \frac{2}{5} = 0.4$



2.2 编程题

In [1]:
import re
from collections import Counter

def preprocess_text(text, n):
    """
    对文本进行预处理，并生成 n-gram 特征和标签。
    参数:
        text: str，输入文本
        n: int，滑动窗口长度
    返回:
        vocab: dict，词到ID的映射（按频率降序）
        features: list of list of str，每个特征窗口包含 n 个词
        labels: list of str，每个窗口对应的下一个词（忽略无后续词的窗口）
    """
    # 1. 小写，去除标点（保留字母和空格）
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)  # 只保留小写字母和空格
    
    # 2. 按空格分词，去除空字符串
    words = text.split()
    
    # 3. 构建词汇表：按出现频率排序，分配ID从0开始
    freq = Counter(words)
    # 按频率降序，频率相同按字母序（可选）
    sorted_words = sorted(freq.keys(), key=lambda w: (-freq[w], w))
    vocab = {word: idx for idx, word in enumerate(sorted_words)}
    
    # 4. 滑动窗口生成特征和标签
    features = []
    labels = []
    for i in range(len(words) - n):
        # 窗口内 n 个词
        feat = words[i:i+n]
        # 标签为下一个词
        label = words[i+n]
        features.append(feat)
        labels.append(label)
    
    return vocab, (features, labels)

# 示例测试
if __name__ == "__main__":
    text = "The time machine"
    n = 2
    vocab, (features, labels) = preprocess_text(text, n)
    print("词汇表:", vocab)
    print("特征:", features)
    print("标签:", labels)

词汇表: {'machine': 0, 'the': 1, 'time': 2}
特征: [['the', 'time']]
标签: ['machine']


## 3 循环神经网络

### 3.1 理论计算题

考虑线性 RNN（无偏置）：
$$h_t = W_{hh} h_{t-1} + W_{hx} x_t, \quad o_t = W_{oh} h_t$$

损失函数：
$$L = \frac{1}{2} \sum_{t=1}^T (o_t - y_t)^2$$

定义误差项 $\delta_t = \frac{\partial L}{\partial h_t}$，由链式法则：
$$\delta_T = W_{oh}^T (o_T - y_T)$$

对于 $t < T$：
$$\delta_t = W_{oh}^T (o_t - y_t) + W_{hh}^T \delta_{t+1}$$

参数 $W_{hh}$ 的梯度为所有时间步局部梯度的和：
$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^T \delta_t \, h_{t-1}^T$$

展开 $\delta_t$ 可得：
$$\delta_t = \sum_{k=t}^T (W_{hh}^T)^{k-t} W_{oh}^T (o_k - y_k)$$

**梯度消失或爆炸的条件**：
- 若 $W_{hh}$ 的谱半径 $> 1$，则 $\| (W_{hh}^T)^{T-t} \|$ 随 $T-t$ 指数增长，导致**梯度爆炸**；
- 若谱半径 $< 1$，则梯度指数衰减，导致**梯度消失**。

---



3.2 编程题
实现 RNN 单元的前向和反向传播（使用 tanh 激活）。

In [2]:
import numpy as np

def rnn_step_forward(x_t, prev_h, W_hh, W_hx, b):
    """
    前向传播：计算当前隐藏状态
    参数:
        x_t: (batch_size, input_size)
        prev_h: (batch_size, hidden_size)
        W_hh: (hidden_size, hidden_size)
        W_hx: (input_size, hidden_size)
        b: (hidden_size,)
    返回:
        h_t: (batch_size, hidden_size)
        cache: 用于反向传播的中间变量
    """
    # 线性变换
    a = prev_h @ W_hh + x_t @ W_hx + b   # (batch, hidden)
    h_t = np.tanh(a)
    cache = (x_t, prev_h, W_hh, W_hx, b, a, h_t)
    return h_t, cache

def rnn_step_backward(dh_next, cache):
    """
    反向传播：已知损失对 h_t 的梯度 dh_next，计算其他梯度
    参数:
        dh_next: (batch_size, hidden_size)
        cache: 前向传播保存的中间变量
    返回:
        dx_t: (batch_size, input_size)
        dh_prev: (batch_size, hidden_size)
        dW_hh: (hidden_size, hidden_size)
        dW_hx: (input_size, hidden_size)
        db: (hidden_size,)
    """
    x_t, prev_h, W_hh, W_hx, b, a, h_t = cache
    
    # tanh 的导数：1 - tanh^2
    da = dh_next * (1 - h_t ** 2)   # (batch, hidden)
    
    # 梯度
    dx_t = da @ W_hx.T              # (batch, input)
    dh_prev = da @ W_hh.T           # (batch, hidden)
    dW_hx = x_t.T @ da              # (input, hidden)
    dW_hh = prev_h.T @ da           # (hidden, hidden)
    db = np.sum(da, axis=0)         # (hidden,)
    
    return dx_t, dh_prev, dW_hh, dW_hx, db

# 简单测试
if __name__ == "__main__":
    batch_size, input_size, hidden_size = 2, 3, 4
    x_t = np.random.randn(batch_size, input_size)
    prev_h = np.random.randn(batch_size, hidden_size)
    W_hh = np.random.randn(hidden_size, hidden_size)
    W_hx = np.random.randn(input_size, hidden_size)
    b = np.random.randn(hidden_size)
    
    h_t, cache = rnn_step_forward(x_t, prev_h, W_hh, W_hx, b)
    dh_next = np.random.randn(batch_size, hidden_size)
    dx_t, dh_prev, dW_hh, dW_hx, db = rnn_step_backward(dh_next, cache)
    
    print("h_t shape:", h_t.shape)
    print("dx_t shape:", dx_t.shape)
    print("dh_prev shape:", dh_prev.shape)
    print("dW_hh shape:", dW_hh.shape)
    print("dW_hx shape:", dW_hx.shape)
    print("db shape:", db.shape)

h_t shape: (2, 4)
dx_t shape: (2, 3)
dh_prev shape: (2, 4)
dW_hh shape: (4, 4)
dW_hx shape: (3, 4)
db shape: (4,)


## 4 高级循环神经网络

### 4.1 理论计算题

深度双向 RNN，共 $L$ 层，每层隐藏单元数 $H$，输入维度 $D$，忽略输出层。

- **第 1 层**：输入维度 $D$，输出维度 $H$。每个方向参数：输入权重 $D \times H$，隐藏权重 $H \times H$，偏置 $H$。两个方向总计：$2 \times (DH + H^2 + H)$

- **第 $l$ 层（$l \ge 2$）**：输入为前一层的两个方向输出拼接，维度 $2H$，输出 $H$。每个方向参数：输入权重 $2H \times H$，隐藏权重 $H \times H$，偏置 $H$。两个方向总计：$2 \times (2H^2 + H^2 + H) = 6H^2 + 2H$

**总参数数量**：
$$\text{Params} = 2(DH + H^2 + H) + (L-1)(6H^2 + 2H)$$

---



4.2 编程题
实现双向 RNN 编码器，使用 torch.nn.RNN。

In [3]:
import torch
import torch.nn as nn

class BidirectionalRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.rnn = nn.RNN(input_dim, hidden_dim, num_layers, 
                          batch_first=False, bidirectional=True)
        self.hidden_dim = hidden_dim
        
    def forward(self, X):
        """
        X: (seq_len, batch, input_dim)
        返回:
            outputs: (seq_len, batch, 2*hidden_dim)  每个时间步拼接后的隐藏状态
            final_state: (batch, 2*hidden_dim)       最终时间步的拼接隐藏状态（序列表示）
        """
        outputs, hn = self.rnn(X)   # outputs: (seq_len, batch, num_directions * hidden_dim)
        # 最终时间步的隐藏状态：取 outputs 的最后一个时间步
        final_state = outputs[-1]   # (batch, 2*hidden_dim)
        return outputs, final_state

# 测试
if __name__ == "__main__":
    seq_len, batch, input_dim = 5, 3, 4
    hidden_dim = 8
    X = torch.randn(seq_len, batch, input_dim)
    encoder = BidirectionalRNNEncoder(input_dim, hidden_dim)
    outputs, final_state = encoder(X)
    print("outputs shape:", outputs.shape)      # (5,3,16)
    print("final_state shape:", final_state.shape)  # (3,16)

outputs shape: torch.Size([5, 3, 16])
final_state shape: torch.Size([3, 16])


## 5 嵌入向量

### 5.1 理论计算题

Skip-gram 负采样。给定中心词 $w_c$ 和上下文词 $w_o$，词向量分别为 $\mathbf{v}_c$（输入向量）和 $\mathbf{u}_o$（输出向量）。从噪声分布 $P_n(w)$ 中采样 $K$ 个负样本 $w_{n_1}, \dots, w_{n_K}$，对应的输出向量为 $\mathbf{u}_{n_k}$。

最小化的负对数似然目标函数为：
$$J = -\log \sigma(\mathbf{v}_c^\top \mathbf{u}_o) - \sum_{k=1}^K \log \sigma(-\mathbf{v}_c^\top \mathbf{u}_{n_k})$$

其中 $\sigma(x) = 1/(1+e^{-x})$。

**噪声分布采样**：负样本从噪声分布 $P_n(w)$ 中抽取，通常取为词频的 $3/4$ 次方（unigram distribution），即 $P_n(w) \propto \text{freq}(w)^{3/4}$，以降低高频词的采样概率，提高低频词被采样的机会。

---



5.2 编程题
实现 CBOW 前向传播和损失计算（完整 softmax）。

In [4]:
import torch
import torch.nn.functional as F

def cbow_forward(context_indices, W, W_out):
    """
    参数:
        context_indices: (batch_size, context_size) 每个样本的上下文词索引
        W: (vocab_size, embedding_dim) 输入权重矩阵
        W_out: (embedding_dim, vocab_size) 输出权重矩阵
    返回:
        loss: 标量，交叉熵损失（平均 batch 损失）
    """
    batch_size, context_size = context_indices.shape
    # 取嵌入向量 (batch, context_size, emb_dim)
    emb = W[context_indices]          # (batch, context_size, emb_dim)
    # 平均上下文向量
    h = emb.mean(dim=1)               # (batch, emb_dim)
    # 输出 logits
    logits = h @ W_out                # (batch, vocab_size)
    # softmax 得到概率分布
    probs = F.softmax(logits, dim=-1) # (batch, vocab_size)
    # 假设目标为中心词索引，但函数未传入 target，我们无法计算损失。
    # 按照要求，需要传入目标中心词索引 target。修改函数签名。
    return probs

def cbow_loss(context_indices, target_indices, W, W_out):
    """
    计算 CBOW 损失。
    参数:
        context_indices: (batch_size, context_size)
        target_indices: (batch_size,) 每个样本的中心词索引
        W, W_out: 同上
    返回:
        loss: 标量，平均交叉熵损失
    """
    batch_size = context_indices.shape[0]
    emb = W[context_indices]          # (batch, context_size, emb_dim)
    h = emb.mean(dim=1)               # (batch, emb_dim)
    logits = h @ W_out                # (batch, vocab_size)
    loss = F.cross_entropy(logits, target_indices)  # 内置交叉熵，包含 softmax
    return loss

# 测试
if __name__ == "__main__":
    vocab_size, emb_dim = 10, 5
    batch_size, context_size = 4, 3
    W = torch.randn(vocab_size, emb_dim, requires_grad=True)
    W_out = torch.randn(emb_dim, vocab_size, requires_grad=True)
    context = torch.randint(0, vocab_size, (batch_size, context_size))
    target = torch.randint(0, vocab_size, (batch_size,))
    loss = cbow_loss(context, target, W, W_out)
    print("CBOW loss:", loss.item())
    loss.backward()
    print("梯度已计算")

CBOW loss: 1.9218391180038452
梯度已计算


## 6 注意力机制

### 6.1 理论计算题

给定：
$$Q \in \mathbb{R}^{2 \times 4}, \quad K \in \mathbb{R}^{3 \times 4}, \quad V \in \mathbb{R}^{3 \times 5}$$

缩放点积注意力，$d_k = 4$，$\sqrt{d_k} = 2$。

**步骤 1：计算得分矩阵**
$$S = \frac{Q K^\top}{\sqrt{d_k}} = \frac{1}{2} Q K^\top \quad \in \mathbb{R}^{2 \times 3}$$

**步骤 2：Softmax 得到注意力权重**
对 $S$ 的每一行进行 Softmax 操作：
$$A = \text{softmax}(S) \quad \in \mathbb{R}^{2 \times 3}, \qquad A_{i,j} = \frac{\exp(S_{i,j})}{\sum_{l=1}^3 \exp(S_{i,l})}$$

**步骤 3：加权求和得到输出**
$$\text{Output} = A V \quad \in \mathbb{R}^{2 \times 5}$$


6.2 编程题
实现多头注意力前向传播。

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.d_v = d_model // num_heads
        
        # 线性投影层
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        
    def forward(self, X):
        """
        X: (seq_len, batch, d_model)
        返回:
            out: (seq_len, batch, d_model)
        """
        seq_len, batch, _ = X.shape
        
        # 线性投影得到 Q, K, V，形状 (seq_len, batch, d_model)
        Q = self.W_q(X)
        K = self.W_k(X)
        V = self.W_v(X)
        
        # 拆分为多头： (seq_len, batch, num_heads, d_k) 然后交换维度
        Q = Q.view(seq_len, batch, self.num_heads, self.d_k).transpose(0, 1)  # (batch, seq_len, num_heads, d_k)
        K = K.view(seq_len, batch, self.num_heads, self.d_k).transpose(0, 1)  # (batch, seq_len, num_heads, d_k)
        V = V.view(seq_len, batch, self.num_heads, self.d_v).transpose(0, 1)  # (batch, seq_len, num_heads, d_v)
        
        # 缩放点积注意力
        # 将 heads 和 batch 合并，或者分开计算，这里我们用合并方式
        # 转置为 (batch, num_heads, seq_len, d_k) 以便计算
        Q = Q.transpose(1, 2)  # (batch, num_heads, seq_len, d_k)
        K = K.transpose(1, 2)  # (batch, num_heads, seq_len, d_k)
        V = V.transpose(1, 2)  # (batch, num_heads, seq_len, d_v)
        
        # 计算得分
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)  # (batch, num_heads, seq_len, seq_len)
        attn_weights = F.softmax(scores, dim=-1)  # (batch, num_heads, seq_len, seq_len)
        
        # 加权求和
        context = torch.matmul(attn_weights, V)  # (batch, num_heads, seq_len, d_v)
        
        # 合并多头： (batch, seq_len, num_heads, d_v) -> (batch, seq_len, d_model)
        context = context.transpose(1, 2).contiguous()  # (batch, seq_len, num_heads, d_v)
        context = context.view(batch, seq_len, self.d_model)  # (batch, seq_len, d_model)
        
        # 最终线性层
        out = self.W_o(context)  # (batch, seq_len, d_model)
        
        # 转回 (seq_len, batch, d_model)
        out = out.transpose(0, 1)  # (seq_len, batch, d_model)
        return out

# 测试
if __name__ == "__main__":
    d_model = 4
    num_heads = 2
    seq_len, batch = 3, 2
    X = torch.randn(seq_len, batch, d_model)
    mha = MultiHeadAttention(d_model, num_heads)
    out = mha(X)
    print("Output shape:", out.shape)  # (3,2,4)

Output shape: torch.Size([3, 2, 4])
